In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec

In [ ]:
#  ***** Pinecone_인덱스생성.png 확인하기 *****

# API Key 설정
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart-index"

# 인덱스가 없는 경우 새로 생성 (OpenAI text-embedding-3-small 기준 1536차원)
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 인덱스 연결 
index = pc.Index(index_name) 

In [ ]:
# --------------- 한개 테스트 ---------------

# 예시 벡터 데이터
# dummy_vector = [0.01] * 1536

# vectors = [
#     {
#         "id": "doc1",
#         "values": dummy_vector,
#         "metadata": {
#             "text": "Pinecone은 벡터 데이터베이스입니다.",
#             "category": "tech"
#         }
#     }
# ]

# Pinecone에 데이터 저장
# index.upsert(
#     vectors=vectors,
#     namespace="example-namespace"
# )

In [ ]:
# ***** Pinecone 인덱스 Upsert 및 유사도 검색 테스트 *****

import os
import time
from openai import OpenAI

# OpenAI 클라이언트 생성
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 테스트할 문서
documents = [
    {
        "id": "doc1",
        "text": "Pinecone은 벡터 데이터베이스입니다."
    },
    {
        "id": "doc2",
        "text": "Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다."
    },
]

# 2. 문장을 임베딩 벡터로 변환
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document["text"] for document in documents],
)

# Pinecone에 저장할 형태로 변환
vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {
            "text": document["text"]
        },
    }
    for document, embedding in zip(documents, response.data)
]

# 3. Pinecone에 저장
# Upsert = Update + Insert
# 같은 id가 이미 있으면 오류 없이 기존 데이터를 덮어씀
index.upsert(vectors=vectors)

# 저장 반영 대기
time.sleep(2)

# 4. 검색할 질문
question = "임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?"

# 질문도 같은 임베딩 모델로 벡터 변환
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# 5. 질문 벡터와 가장 유사한 문서 검색
result = index.query(
    vector=question_vector,
    top_k=2,
    include_metadata=True,
)

# 6. 결과 출력
print(f"질문: {question}\n")

for match in result.matches:
    print(
        f"{match.id}: "
        f"{match.metadata['text']} "
        f"(유사도: {match.score:.4f})"
    )

In [11]:
# 문장을 바꿔서 Pinecone 유사도 검색 테스트

documents = [
    {"id": "doc1", "text": "고양이는 조용하고 독립적인 반려동물입니다."},
    {"id": "doc2", "text": "강아지는 사람과 친밀하게 지내는 반려동물입니다."},
    {"id": "doc3", "text": "자동차는 사람을 이동시키는 교통수단입니다."},
]

# 문서 임베딩 생성
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document["text"] for document in documents],
)

vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {"text": document["text"]},
    }
    for document, embedding in zip(documents, response.data)
]

# Pinecone에 저장
index.upsert(vectors=vectors)
time.sleep(2)

# 검색 질문
question = "사람과 친하게 지내는 반려동물은 무엇인가요?"

# 질문 임베딩 생성
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# 유사도 검색
result = index.query(
    vector=question_vector,
    top_k=3,
    include_metadata=True,
)

print(f"질문: {question}\n")

for match in result.matches:
    print(f"{match.id}: {match.metadata['text']} (유사도: {match.score:.4f})")

질문: 사람과 친하게 지내는 반려동물은 무엇인가요?

doc2: 강아지는 사람과 친밀하게 지내는 반려동물입니다. (유사도: 0.6179)
doc3: 자동차는 사람을 이동시키는 교통수단입니다. (유사도: 0.2525)
doc1: 고양이는 조용하고 독립적인 반려동물입니다. (유사도: 0.2495)
